In [1]:
import pandas as pd
from sqlalchemy import create_engine, inspect, text

engine = create_engine("sqlite:///sdmdb.db")
inspector = inspect(engine)

tables = inspector.get_table_names()

data = {}

#alles verwijderen
with engine.begin() as conn:
    for table in tables:
        try:
            conn.execute(text(f"DELETE FROM {table}"))
        except Exception as e:
            print(f"fout bij delete {table}: {e}")

#check
for table in tables:
    try:
        query = f"SELECT * FROM {table}"
        data[table] = pd.read_sql_query(query, engine)
    except Exception as e:
        print(f"fout bij load {table}: {e}")

print("\n✅ klaar: reset + reload compleet")


✅ klaar: reset + reload compleet


Reset SDM

In [3]:
import sqlite3
import logging
import os
os.makedirs("logs", exist_ok=True)

logging.basicConfig( 
    filename="logs/sdm_de.log",
    level=logging.INFO, 
    format="%(asctime)s | %(name)s |  %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H-%M-%S",
    force=True
    )

logger = logging.getLogger("DWH")
logger.setLevel(logging.INFO)

sdm_conn = sqlite3.connect('sdmdb.db')
sdm_cursor = sdm_conn.cursor()

def resetSDM():
    """
    Verwijdert alle tabellen uit het SDM
    + logging
    """

    try:
        logger.info("SDM|RESET_SDM_START|Data/SDM.db|||||START")

        sdm_cursor.execute(
            "PRAGMA foreign_keys = OFF;"
        )

        tables = sdm_cursor.execute("""
            SELECT name
            FROM sqlite_master
            WHERE type='table'
        """).fetchall()

        if not tables:
            logger.warning("SDM|ERROR|Data/SDM.db||||FAILED|%s",str(e))

        for (table,) in tables:

            try:
                sdm_cursor.execute(
                    f"DROP TABLE IF EXISTS {table}"
                )

                logger.info(
                "SDM|READ|Data/SDM.db|%s|||SUCCESS",
                table
                )

                print(f"Dropped: {table}")

            except Exception as table_error:

                logger.error(
                f"SDM|ERROR|Data/SDM.db|%s|||FAILED|%s",
                table,
                str(e)
                )

        sdm_conn.commit()

        logger.info("SDM|LOAD_SDM_END|Data/SDM.db|||||END")
        print("SDM volledig gereset (DROP)")

    except Exception as e:

        sdm_conn.rollback()

        logger.error(
            f"logger.error("
            "SDM|ERROR|Data/SDM.db||||FAILED|%s",
            str(e)
            )

        print(f"ERROR: {e}")

    finally:

        sdm_cursor.execute(
            "PRAGMA foreign_keys = ON;"
        )

        logger.info("SDM|RESET_SDM_END|%s|||||END")


# RUN
resetSDM()

Dropped: age_group
Dropped: customer_segment
Dropped: customer_type
Dropped: sales_territory
Dropped: crm_country
Dropped: customer_headquarters
Dropped: sales_demographic
Dropped: customer
Dropped: customer_store
Dropped: customer_contact
Dropped: product_line
Dropped: product_type
Dropped: product
Dropped: go_sales_country
Dropped: retailer_site
Dropped: sales_branch
Dropped: order_method
Dropped: sales_staff
Dropped: order_header
Dropped: return_reason
Dropped: order_details
Dropped: returned_item
Dropped: sales_office
Dropped: sales_representative
Dropped: satisfaction_type
Dropped: satisfaction
Dropped: course
Dropped: training
Dropped: inventory_level
Dropped: product_forecast
Dropped: sales_target
SDM volledig gereset (DROP)
